# Notebook 2 - Baseline de búsqueda léxica con TF-IDF

## Objetivo

En este notebook se implementa un sistema de recuperación de películas basado en un enfoque léxico tradicional utilizando TF-IDF.

Cada película será representada a partir de su columna `search_text`, construida en el notebook anterior como una descripción textual unificada con información relevante de la película.

A partir de esa representación:

- se transformará el corpus en una matriz TF-IDF
- se representarán consultas del usuario en el mismo espacio vectorial
- se calculará la similitud entre consulta y películas mediante cosine similarity
- se devolverá un ranking de resultados relevantes

Este sistema funcionará como baseline para comparar más adelante con un enfoque de búsqueda semántica basado en embeddings multilingües.

In [88]:
# Manejo de datos
import pandas as pd
import numpy as np

# Vectorización TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Similitud entre vectores
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
# Cargar el dataset - origen repo público proyecto en github

url = "https://raw.githubusercontent.com/jiimedina/movie-semantic-search/refs/heads/main/data/movies_mini.csv"
df = pd.read_csv(url)
df.head()

# url = "https://raw.githubusercontent.com/jiimedina/movie-semantic-search/refs/heads/main/data/movies_mini_v2.csv"
# df = pd.read_csv(url)
# df.head()


,id,title,overview,genres,keywords,cast,search_text
0,893694,Eva,When Eva gets involved in a steamy threesome w...,"['Drama', 'Romance']",[],"['Angeli Khang', 'Ava Mendez', 'Marco Gomez', ...",Eva | When Eva gets involved in a steamy three...
1,98,Gladiator,"After the death of Emperor Marcus Aurelius, hi...","['Action', 'Drama', 'Adventure']","['gladiator', 'rome, italy', 'arena', 'senate'...","['Russell Crowe', 'Joaquin Phoenix', 'Connie N...",Gladiator | After the death of Emperor Marcus ...
2,609681,The Marvels,When her duties send her to an anomalous wormh...,"['Science Fiction', 'Adventure', 'Action']","['hero', 'superhero', 'space travel', 'based o...","['Brie Larson', 'Teyonah Parris', 'Iman Vellan...",The Marvels | When her duties send her to an a...
3,605,The Matrix Revolutions,The human city of Zion defends itself against ...,"['Adventure', 'Action', 'Thriller', 'Science F...","['dying and death', 'rescue', 'future', 'missi...","['Keanu Reeves', 'Laurence Fishburne', 'Carrie...",The Matrix Revolutions | The human city of Zio...
4,1306368,The Rip,Trust frays when a team of Miami cops discover...,"['Action', 'Thriller', 'Crime']","['miami, florida', 'police', 'investigation', ...","['Matt Damon', 'Ben Affleck', 'Teyana Taylor',...",The Rip | Trust frays when a team of Miami cop...


## Inspección inicial del dataset

Antes de aplicar el modelo TF-IDF, se realiza una exploración básica del dataset para:

- verificar la estructura de los datos
- identificar posibles valores nulos
- inspeccionar la columna `search_text`
- asegurar que los textos sean adecuados para la vectorización

Este paso es fundamental para garantizar la calidad del pipeline de recuperación.

In [90]:
df.shape

(20, 7)

In [91]:
df.columns

Index(['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'search_text'], dtype='object')

In [92]:
# Revisar valores nulos en search_text
df["search_text"].isna().sum()

np.int64(0)

In [93]:
# Ver ejemplos reales de texto
df["search_text"].head(3).tolist()

['Eva | When Eva gets involved in a steamy threesome with a houseboy and her lady boss, she realizes she has to choose only one between them. | Genres: Drama, Romance | Cast: Angeli Khang, Ava Mendez, Marco Gomez, Ivan Padilla, Angelica Cervantes',
 "Gladiator | After the death of Emperor Marcus Aurelius, his devious son takes power and demotes Maximus, one of Rome's most capable generals who Marcus preferred. Eventually, Maximus is forced to become a gladiator and battle to the death against other men for the amusement of paying audiences. | Genres: Action, Drama, Adventure | Keywords: gladiator, rome, italy, arena, senate, roman empire, parent child relationship, emperor, slavery, ancient rome, revenge, battlefield, slave auction, historical fiction, ancient world, combat, chariot, philosopher, grim, barbarian horde, 2nd century, successor, commodus, maximus, excited, gladiador | Cast: Russell Crowe, Joaquin Phoenix, Connie Nielsen, Oliver Reed, Richard Harris",
 "The Marvels | When 

### Observaciones

La columna `search_text` contiene una representación enriquecida de cada película, combinando:

- título
- descripción (overview)
- géneros
- keywords
- reparto

Esto permite que el modelo TF-IDF capture distintos aspectos semánticos de las películas.

No se detectaron valores nulos ni textos vacíos, por lo que el dataset se considera apto para la vectorización.

En futuras iteraciones, podrían explorarse variantes en la construcción de esta columna para mejorar la calidad de la recuperación.

## Limpieza básica del texto

Se realiza una limpieza mínima sobre la columna `search_text`:

- reemplazo de valores nulos
- eliminación de textos vacíos

No se aplican técnicas avanzadas de preprocesamiento (como lematización o stemming), ya que el objetivo es construir un baseline simple basado en TF-IDF.

In [94]:
# Reemplazar nulos por string vacío
df["search_text"] = df["search_text"].fillna("")

# Eliminar filas donde el texto esté vacío o solo tenga espacios
df = df[df["search_text"].str.strip() != ""].copy()

# Verificar resultado
df.shape

(20, 7)

Aunque no se detectaron valores nulos en la inspección inicial, se incluye este paso de limpieza para asegurar la robustez del pipeline ante posibles cambios en el dataset.

## Representación del texto con TF-IDF

Para poder comparar consultas con películas, es necesario transformar el texto en una representación numérica.

Se utiliza TF-IDF (Term Frequency - Inverse Document Frequency), una técnica clásica de recuperación de información que asigna mayor peso a los términos que:

- son frecuentes dentro de un documento
- pero poco frecuentes en el resto del corpus

De esta forma, cada película queda representada como un vector en un espacio de características construido a partir del vocabulario del corpus.

Este enfoque corresponde a un modelo de búsqueda léxico, basado en coincidencias de términos.

## Construcción de la matriz TF-IDF

Se utiliza `TfidfVectorizer` de `scikit-learn` para transformar la columna `search_text` en una matriz numérica.

Configuración utilizada:

- `lowercase=True`: normaliza el texto a minúsculas
- `stop_words="english"`: elimina palabras vacías en inglés
- `max_features=5000`: limita el tamaño del vocabulario

La eliminación de stopwords se justifica porque el corpus está en inglés y estas palabras suelen aportar poco valor semántico.

In [95]:
# Inicializar vectorizador TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000
)

# Crear matriz TF-IDF
tfidf_matrix = vectorizer.fit_transform(df["search_text"])

## Inspección de la matriz TF-IDF

Una vez construida la matriz, se analiza su forma y tipo para entender cómo se representan los datos.

In [96]:
print("Shape de la matriz TF-IDF:", tfidf_matrix.shape)

Shape de la matriz TF-IDF: (20, 913)


In [97]:
type(tfidf_matrix)

scipy.sparse._csr.csr_matrix

La matriz TF-IDF tiene dimensión (20, 913), lo que indica que:

- cada fila representa una película
- cada columna representa un término del vocabulario

El tamaño del vocabulario es menor al máximo definido (5000) debido a que el dataset es reducido.

Además, la matriz es de tipo disperso (`csr_matrix`), lo cual es esperado en este tipo de representaciones, ya que la mayoría de los términos no aparece en todos los documentos.

## Exploración del vocabulario

Se inspeccionan algunos de los términos utilizados por el modelo para representar los documentos.

In [98]:
feature_names = vectorizer.get_feature_names_out()
feature_names[:30]

array(['10', '1392', '1453', '15th', '1910', '2nd', '476', 'abruptly',
       'absurd', 'abundance', 'accompanied', 'action', 'actor', 'adopted',
       'adventure', 'adversary', 'affleck', 'aftercreditsstinger',
       'aftermath', 'agency', 'agent', 'ages', 'aging', 'agrees', 'aka',
       'alien', 'ally', 'amid', 'amnesia', 'amusement'], dtype=object)

In [99]:
len(vectorizer.get_feature_names_out())

913

El vocabulario contiene 913 términos.

Entre ellos se observan:

- términos relevantes como géneros (`action`, `adventure`)
- palabras descriptivas del contenido (`alien`, `amnesia`)
- nombres propios (actores)
- algunos valores numéricos (años o referencias históricas)

Esto indica que el modelo captura información útil, aunque también incluye ciertos elementos que podrían considerarse ruido en futuras mejoras.

## Análisis de representación de una película

Se analiza cómo una película específica es representada en el espacio TF-IDF.

Esto permite entender qué términos son considerados más relevantes por el modelo.

In [100]:
#Veo cuántos términos tiene una pelicula
tfidf_matrix[0].nnz

24

La película utiliza 24 términos del vocabulario total.

Esto refleja la naturaleza dispersa de la representación, donde cada documento contiene solo una pequeña fracción de los términos disponibles.

In [101]:
#Veo las palabras que tienen mayor peso en una pelicula
doc_index = 0
row = tfidf_matrix[doc_index].toarray().flatten()

top_term_indices = row.argsort()[::-1][:10]

[(feature_names[i], row[i]) for i in top_term_indices if row[i] > 0]

[('eva', np.float64(0.4118855506935343)),
 ('cervantes', np.float64(0.20594277534676714)),
 ('gets', np.float64(0.20594277534676714)),
 ('steamy', np.float64(0.20594277534676714)),
 ('choose', np.float64(0.20594277534676714)),
 ('boss', np.float64(0.20594277534676714)),
 ('threesome', np.float64(0.20594277534676714)),
 ('mendez', np.float64(0.20594277534676714)),
 ('padilla', np.float64(0.20594277534676714)),
 ('houseboy', np.float64(0.20594277534676714))]

Se observa que los términos con mayor peso incluyen:

- el nombre de la película
- nombres propios del elenco
- palabras clave del contenido

Esto indica que TF-IDF logra capturar características distintivas de cada película.

Sin embargo, el modelo se basa exclusivamente en coincidencias de términos y no comprende relaciones semánticas entre palabras.

## Interpretabilidad del modelo TF-IDF

Una ventaja del enfoque TF-IDF es su interpretabilidad.

A diferencia de modelos más complejos, permite analizar:

- qué términos representan cada documento
- qué peso tiene cada término
- qué características influyen en la similitud

Esto facilita comprender el comportamiento del modelo y detectar posibles mejoras.

## Limitaciones del enfoque TF-IDF

El modelo presenta algunas limitaciones importantes:

- no captura significado semántico
- depende de coincidencias exactas de palabras
- no reconoce sinónimos
- no maneja bien consultas en diferentes idiomas

Estas limitaciones motivan el uso posterior de embeddings en el proyecto.

## Conclusión de la representación TF-IDF

Se construyó una representación vectorial del corpus que permite comparar películas en función de los términos que contienen.

Esta representación será utilizada en la siguiente etapa para calcular similitudes entre consultas y documentos mediante cosine similarity.

## Medida de similitud: Cosine Similarity

Una vez que las películas y las consultas están representadas como vectores TF-IDF, es necesario definir una forma de medir qué tan similares son entre sí.

Se utiliza la similitud coseno (cosine similarity), que mide el ángulo entre dos vectores en el espacio vectorial.

- Un valor cercano a 1 indica alta similitud
- Un valor cercano a 0 indica baja similitud

Esta medida es ampliamente utilizada en tareas de recuperación de información, ya que permite comparar documentos independientemente de su longitud. Esto permite rankear las películas según su relevancia respecto a la consulta.

## Primera consulta de prueba

Se realiza una consulta inicial en inglés para observar el funcionamiento del sistema.

Se elige una query con términos presentes en el corpus para facilitar la interpretación de los resultados.

In [102]:
query = "space adventure aliens"

# Transformar la query al espacio TF-IDF
query_vector = vectorizer.transform([query])

# Calcular similitud entre la query y todas las películas
similarities = cosine_similarity(query_vector, tfidf_matrix)

In [103]:
similarities.shape

(1, 20)

El resultado tiene dimensión (1, 20):

- 1 corresponde a la query
- 20 corresponde a cada película del dataset

Cada valor representa qué tan similar es la película respecto a la consulta.

## Ranking de resultados

Se ordenan las películas según su similitud con la consulta, de mayor a menor, para obtener los resultados más relevantes.

In [104]:
similarity_scores = similarities.flatten()

# Obtener índices ordenados (de mayor a menor similitud)
top_indices = similarity_scores.argsort()[::-1][:5]

df.iloc[top_indices][["title", "search_text"]]

,title,search_text
2,The Marvels,The Marvels | When her duties send her to an a...
11,Predator: Badlands,"Predator: Badlands | Cast out from his clan, a..."
7,The Lord of the Rings: The War of the Rohirrim,The Lord of the Rings: The War of the Rohirrim...
17,The Wild Robot,"The Wild Robot | After a shipwreck, an intelli..."
3,The Matrix Revolutions,The Matrix Revolutions | The human city of Zio...


Se utiliza `argsort()` para ordenar los índices según los valores de similitud.

Luego se invierte el orden para obtener las películas más similares primero.

Finalmente, se seleccionan las 5 películas con mayor score.

In [105]:
results = df.iloc[top_indices][["title"]].copy()
results["score"] = similarity_scores[top_indices]
results

,title,score
2,The Marvels,0.274096
11,Predator: Badlands,0.043113
7,The Lord of the Rings: The War of the Rohirrim,0.039980
17,The Wild Robot,0.039122
3,The Matrix Revolutions,0.030916


### Análisis de los scores de similitud

El sistema recupera como resultado principal la película "The Marvels", con un score superior al resto.

Esto indica que existe una coincidencia léxica relevante entre la consulta y el documento.

Los demás resultados presentan scores menores, lo que sugiere una similitud más débil.

In [106]:
query_vector_array = query_vector.toarray().flatten()

[(feature_names[i], query_vector_array[i])
 for i in query_vector_array.argsort()[::-1] if query_vector_array[i] > 0]

[('space', np.float64(0.8626438810053888)),
 ('adventure', np.float64(0.5058117580325318))]

### Inspección de la representación de la consulta

Se observa que la consulta queda representada principalmente por los términos `space` y `adventure`.

Esto indica que no todos los términos originales de la query forman parte del vocabulario del modelo.

A pesar de esto, los términos presentes son suficientemente informativos para recuperar resultados relevantes, como se observa en el ranking obtenido.

En este caso, la consulta permite recuperar resultados razonables, lo que indica que el modelo funciona adecuadamente cuando existe coincidencia léxica entre la query y los documentos.

A continuación, se analizan distintos tipos de consultas para evaluar el comportamiento del modelo en escenarios más variados.

In [107]:
#Función de búsqueda
def search_movies_tfidf(query, df, vectorizer, tfidf_matrix, top_k=5):
    """
    Devuelve las top_k películas más similares a una consulta
    usando TF-IDF + cosine similarity.
    """
    # Vectorizar la query
    query_vector = vectorizer.transform([query])

    # Calcular similitud
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Ordenar resultados de mayor a menor similitud
    top_indices = similarity_scores.argsort()[::-1][:top_k]

    # Construir DataFrame de resultados
    results = df.iloc[top_indices].copy()
    results["score"] = similarity_scores[top_indices]

    return results[["title", "score", "search_text"]]

## Análisis cualitativo de consultas

Se evalúa el comportamiento del sistema TF-IDF sobre distintas consultas con el objetivo de analizar sus fortalezas y limitaciones.

Se seleccionan consultas con distintos niveles de dificultad:

- consultas con alto solapamiento léxico
- consultas con coincidencia parcial
- consultas en otro idioma

Esto permite observar cómo responde el modelo en diferentes escenarios.

### Caso 1: consulta con buen solapamiento léxico

Consulta: "space adventure aliens"

In [108]:
search_movies_tfidf("space adventure aliens", df, vectorizer, tfidf_matrix)

,title,score,search_text
2,The Marvels,0.274096,The Marvels | When her duties send her to an a...
11,Predator: Badlands,0.043113,"Predator: Badlands | Cast out from his clan, a..."
7,The Lord of the Rings: The War of the Rohirrim,0.039980,The Lord of the Rings: The War of the Rohirrim...
17,The Wild Robot,0.039122,"The Wild Robot | After a shipwreck, an intelli..."
3,The Matrix Revolutions,0.030916,The Matrix Revolutions | The human city of Zio...


El sistema recupera como resultado principal la película "The Marvels", con un score significativamente mayor al resto.

Esto indica que existe una coincidencia léxica relevante entre la consulta y el documento.

Este caso muestra que TF-IDF funciona adecuadamente cuando la consulta comparte términos relevantes con el corpus.

### Caso 2: coincidencia parcial

Consulta: "roman general revenge"

In [109]:
search_movies_tfidf("roman general revenge", df, vectorizer, tfidf_matrix)

,title,score,search_text
1,Gladiator,0.147242,Gladiator | After the death of Emperor Marcus ...
19,Norbit,0.000000,Norbit | A mild-mannered guy who is engaged to...
17,The Wild Robot,0.000000,"The Wild Robot | After a shipwreck, an intelli..."
18,Mission: Impossible - Fallout,0.000000,Mission: Impossible - Fallout | When an IMF mi...
16,Constantine,0.000000,Constantine | John Constantine has literally b...


El sistema recupera correctamente la película "Gladiator" como primer resultado.

Sin embargo, el score es relativamente bajo y el resto de las películas presentan valores cercanos a 0.

Esto indica que la coincidencia léxica existe, pero la señal es limitada.

### Caso 3: consulta en español

Consulta: "niño mago lucha contra el mal"


In [110]:
search_movies_tfidf("niño mago lucha contra el mal", df, vectorizer, tfidf_matrix)

,title,score,search_text
19,Norbit,0.0,Norbit | A mild-mannered guy who is engaged to...
18,Mission: Impossible - Fallout,0.0,Mission: Impossible - Fallout | When an IMF mi...
17,The Wild Robot,0.0,"The Wild Robot | After a shipwreck, an intelli..."
16,Constantine,0.0,Constantine | John Constantine has literally b...
15,Swan Song,0.0,Swan Song | An aging actor remembers his past ...


El sistema falla completamente: todas las películas presentan un score de 0.

Esto ocurre porque el vocabulario fue aprendido a partir de documentos en inglés, por lo que ninguna de las palabras en español forma parte del espacio vectorial.

Este resultado evidencia una limitación crítica del enfoque: TF-IDF no puede manejar búsquedas multilingües sin vocabulario compartido.

### Conclusiones del análisis cualitativo

El modelo TF-IDF:

- funciona bien cuando hay coincidencia léxica
- puede recuperar resultados correctos con señal débil
- falla completamente sin solapamiento de vocabulario

Esto muestra que el modelo no comprende el significado de las consultas, sino que depende de coincidencias exactas de términos.

Estas limitaciones motivan el uso de modelos semánticos en las siguientes etapas del proyecto.

## Evaluación cuantitativa: Precision@K

Además del análisis cualitativo, se incorpora una evaluación cuantitativa simple mediante la métrica Precision@K.

Esta métrica mide qué proporción de los primeros K resultados recuperados por el sistema son relevantes para una consulta dada.

Dado que se trabaja con un dataset reducido, se definirá manualmente un pequeño conjunto de consultas de prueba junto con sus películas relevantes esperadas.

Esto permite obtener una primera estimación del desempeño del baseline TF-IDF.

In [111]:
# Conjunto de consultas de evaluación y títulos considerados relevantes
evaluation_queries = {
    "space adventure aliens": ["The Marvels"],
    "roman general revenge": ["Gladiator"],
    "superhero space travel": ["The Marvels"]
}

In [112]:
def precision_at_k(retrieved_titles, relevant_titles, k):
    """
    Calcula Precision@K dado un conjunto de títulos recuperados
    y un conjunto de títulos relevantes.
    """
    # Tomar solo los primeros K resultados recuperados
    retrieved_k = retrieved_titles[:k]

    # Convertir los relevantes a conjunto para facilitar la comparación
    relevant_set = set(relevant_titles)

    # Contar cuántos títulos recuperados son relevantes
    relevant_retrieved = sum(title in relevant_set for title in retrieved_k)

    # Calcular la proporción de relevantes en el top K
    return relevant_retrieved / k

### Ejemplo de cálculo para una consulta

Antes de evaluar todas las consultas, se calcula Precision@K para un caso individual con el fin de ilustrar el procedimiento.

In [113]:
# Seleccionar una query de ejemplo
query = "space adventure aliens"
relevant_titles = evaluation_queries[query]

# Recuperar resultados con el buscador TF-IDF
results = search_movies_tfidf(query, df, vectorizer, tfidf_matrix, top_k=5)
retrieved_titles = results["title"].tolist()

# Calcular Precision@1 y Precision@5 para esta query
p_at_1 = precision_at_k(retrieved_titles, relevant_titles, k=1)
p_at_5 = precision_at_k(retrieved_titles, relevant_titles, k=5)

print("Resultados recuperados:", retrieved_titles)
print("Relevantes esperados:", relevant_titles)
print("Precision@1:", p_at_1)
print("Precision@5:", p_at_5)

Resultados recuperados: ['The Marvels', 'Predator: Badlands', 'The Lord of the Rings: The War of the Rohirrim', 'The Wild Robot', 'The Matrix Revolutions']
Relevantes esperados: ['The Marvels']
Precision@1: 1.0
Precision@5: 0.2


### Evaluación sobre el conjunto de consultas

A continuación, se calcula Precision@1, Precision@3 y Precision@5 para todas las consultas definidas.

In [114]:
evaluation_results = []

# Evaluar cada consulta del conjunto de prueba
for query, relevant_titles in evaluation_queries.items():
    results = search_movies_tfidf(query, df, vectorizer, tfidf_matrix, top_k=5)
    retrieved_titles = results["title"].tolist()

    # Calcular métricas en distintas profundidades del ranking
    p_at_1 = precision_at_k(retrieved_titles, relevant_titles, k=1)
    p_at_3 = precision_at_k(retrieved_titles, relevant_titles, k=3)
    p_at_5 = precision_at_k(retrieved_titles, relevant_titles, k=5)

    # Guardar resultados
    evaluation_results.append({
        "query": query,
        "relevant_titles": relevant_titles,
        "retrieved_titles": retrieved_titles,
        "Precision@1": p_at_1,
        "Precision@3": p_at_3,
        "Precision@5": p_at_5
    })

# Convertir resultados a DataFrame para facilitar la lectura
evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

,query,relevant_titles,retrieved_titles,Precision@1,Precision@3,Precision@5
0,space adventure aliens,[The Marvels],"[The Marvels, Predator: Badlands, The Lord of ...",1.0,0.333333,0.2
1,roman general revenge,[Gladiator],"[Gladiator, Norbit, The Wild Robot, Mission: I...",1.0,0.333333,0.2
2,superhero space travel,[The Marvels],"[The Marvels, Constantine, Mission: Impossible...",1.0,0.333333,0.2


### Interpretación de Precision@K

Los resultados muestran el desempeño del sistema en distintas profundidades del ranking:

- Precision@1 evalúa si el primer resultado recuperado es relevante
- Precision@3 mide la proporción de relevantes entre los primeros tres resultados
- Precision@5 extiende este análisis a los primeros cinco

En un dataset pequeño como el utilizado en esta etapa, Precision@1 resulta especialmente informativa, ya que permite observar si el sistema logra ubicar correctamente la película más relevante en la primera posición.

Además, dado que se definió una única película relevante por consulta, el valor máximo posible de Precision@K disminuye al aumentar K.

### Interpretación de los resultados

Los resultados muestran que el sistema alcanza un valor de Precision@1 igual a 1.0 en todas las consultas evaluadas.

Esto indica que, en estos casos, el modelo logra recuperar la película relevante en la primera posición del ranking.

Sin embargo, al analizar Precision@3 y Precision@5, se observa que los valores disminuyen a 0.33 y 0.2 respectivamente. Esto se debe a que solo se definió una película relevante por consulta, por lo que la proporción de resultados relevantes disminuye al aumentar K.

Además, estos resultados sugieren que, aunque el modelo logra identificar correctamente el resultado principal, no discrimina de forma precisa entre el resto de las películas, lo que limita la calidad del ranking más allá de la primera posición.

### Conclusiones de la evaluación

El baseline TF-IDF muestra un buen desempeño en términos de Precision@1, logrando recuperar el resultado más relevante en la primera posición.

No obstante, su desempeño en posiciones posteriores es limitado, lo que refleja una menor capacidad para ordenar correctamente el resto de los resultados.

Este comportamiento es consistente con las características del modelo:

- depende de coincidencias léxicas directas
- no captura relaciones semánticas
- presenta limitaciones en escenarios más complejos o multilingües

Estos resultados sirven como referencia para comparar con el enfoque semántico basado en embeddings en las siguientes etapas del proyecto.

### Observación adicional

Dado que se definió una única película relevante por consulta, Precision@1 resulta ser la métrica más informativa en este contexto.

En escenarios con múltiples documentos relevantes por consulta, métricas como Precision@K o Recall@K permitirían un análisis más completo del sistema.

## Discusión del baseline

El enfoque TF-IDF implementado permite construir un sistema de búsqueda funcional de forma simple y eficiente.

Sin embargo, su desempeño depende fuertemente de la coincidencia exacta de términos entre la consulta y los documentos.

Esto limita su capacidad para:

- capturar sinónimos
- manejar variaciones lingüísticas
- realizar búsquedas en distintos idiomas

Estas limitaciones motivan la incorporación de modelos semánticos en la siguiente etapa del proyecto.

### Interpretación de los resultados

Los resultados muestran que el sistema alcanza un valor de Precision@1 igual a 1.0 en todas las consultas evaluadas.

Esto indica que, en estos casos, el modelo logra recuperar la película relevante en la primera posición del ranking.

Sin embargo, al analizar Precision@3 y Precision@5, se observa que los valores disminuyen a 0.33 y 0.2 respectivamente. Esto se debe a que solo se definió una película relevante por consulta, por lo que la proporción de resultados relevantes disminuye al aumentar K.

Además, estos resultados sugieren que, aunque el modelo logra identificar correctamente el resultado principal, no discrimina de forma precisa entre el resto de las películas, lo que limita la calidad del ranking más allá de la primera posición.

## Discusión del baseline

El enfoque TF-IDF implementado permite construir un sistema de búsqueda funcional de forma simple y eficiente.

Sin embargo, su desempeño depende fuertemente de la coincidencia exacta de términos entre la consulta y los documentos.

Esto limita su capacidad para:

- capturar sinónimos
- manejar variaciones lingüísticas
- realizar búsquedas en distintos idiomas

Estas limitaciones motivan la incorporación de modelos semánticos en la siguiente etapa del proyecto.